# 1. Setup

### 1.1 Install deps & packages

In [1]:
%pip install numpy pandas matplotlib kagglehub
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import kagglehub
import shutil
import os


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


/Users/kaneviggers/Desktop/Transaction-fraud-detection/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1.2 Download dataset

In [2]:
os.mkdir('original_dataset')

path = kagglehub.dataset_download("nafiulislam490/bank-transaction-fraud-detection-dataset", output_dir='original_dataset')

# Remove uesless metadata
shutil.rmtree('original_dataset/.complete/', ignore_errors=True)

print('Dataset downloaded')

100%|██████████| 42.4M/42.4M [05:05<00:00, 145kB/s] 

Extracting files...


Dataset downloaded


# 2. Data pipeline

### 2.1 Load csv into dataframe

In [48]:
fraud_data = pd.read_csv('original_dataset/bank_fraud.csv')

print(f"Shape\n{fraud_data.shape}\n")
print(f"Description\n{fraud_data.describe()}\n")
print(f"Head\n{fraud_data.head()}")

Shape
(1000000, 26)

Description
          hour_of_day      is_weekend  is_night_transaction    customer_age  \
count  1000000.000000  1000000.000000        1000000.000000  1000000.000000   
mean        11.496978        0.286022              0.375057       41.771678   
std          6.923751        0.451900              0.484138       13.424588   
min          0.000000        0.000000              0.000000       18.000000   
25%          5.000000        0.000000              0.000000       32.000000   
50%         11.000000        0.000000              0.000000       42.000000   
75%         18.000000        1.000000              1.000000       51.000000   
max         23.000000        1.000000              1.000000       85.000000   

         credit_score  account_age_years  account_balance  transaction_amount  \
count  1000000.000000     1000000.000000    1000000.00000      1000000.000000   
mean       679.028781           4.987911      16594.25442          204.724665   
std         

# 3. Data Cleansing and Transformation

### 3.1 Dropping columns

Looking at the dataset, we can see the transaction ID is unique to every row, therefore it won't be useful in deriving any value.
We can use the customer contry to identify if the transaction took place in a high risk country, for this reason we can drop the city as it's too spesific.

In [49]:
fraud_data = fraud_data.drop(columns=['transaction_id', 'country', 'city'])

fraud_data.head()

,customer_id,transaction_date,transaction_time,hour_of_day,is_weekend,is_night_transaction,merchant_category,payment_method,device_type,customer_age,...,transaction_amount,num_prev_transactions,transaction_freq_monthly,distance_from_home_km,time_since_last_txn_hrs,is_international,failed_attempts,pin_changed_recently,is_fraud,fraud_type
0,CUST00121959,2023-08-17,21:13:00,21,0,0,Grocery,Bank Transfer,POS Terminal,18,...,39.49,157,23,52.7,10.20,0,0,0,0,NaN
1,CUST00146868,2024-02-06,05:16:00,5,0,1,Healthcare,Cheque,Desktop,30,...,153.71,153,23,0.9,12.47,0,0,0,0,NaN
2,CUST00131933,2024-06-28,12:15:00,12,0,0,Grocery,Crypto,Mobile,20,...,118.20,161,20,9.2,0.08,0,1,0,0,NaN
3,CUST00103695,2023-03-16,02:53:00,2,0,1,Utilities,Debit Card,Mobile,29,...,49.50,160,25,14.8,17.94,1,0,1,1,Synthetic Identity
4,CUST00119880,2024-07-12,12:39:00,12,0,0,Clothing,Debit Card,Desktop,49,...,30.74,134,18,38.9,2.16,0,0,0,0,NaN


### 3.2 Normalising data

Since a lot of these columns are categories we can convert them to a number using a dict

In [50]:
# A list of categorical columns
categorical_columns = ['merchant_category', 'payment_method', 'device_type', 'fraud_type']

# Create a dict of the enum values for each category
category_values = {}

for category in categorical_columns:
    category_values[category] = fraud_data[category].unique().tolist()

for i in range(fraud_data.shape[0]):
    for key in category_values:
        fraud_data.at[i, key] = category_values[key].index(fraud_data.loc[i][key])
    
fraud_data.head()

,customer_id,transaction_date,transaction_time,hour_of_day,is_weekend,is_night_transaction,merchant_category,payment_method,device_type,customer_age,...,transaction_amount,num_prev_transactions,transaction_freq_monthly,distance_from_home_km,time_since_last_txn_hrs,is_international,failed_attempts,pin_changed_recently,is_fraud,fraud_type
0,CUST00121959,2023-08-17,21:13:00,21,0,0,0,0,0,18,...,39.49,157,23,52.7,10.20,0,0,0,0,0
1,CUST00146868,2024-02-06,05:16:00,5,0,1,1,1,1,30,...,153.71,153,23,0.9,12.47,0,0,0,0,0
2,CUST00131933,2024-06-28,12:15:00,12,0,0,0,2,2,20,...,118.20,161,20,9.2,0.08,0,1,0,0,0
3,CUST00103695,2023-03-16,02:53:00,2,0,1,2,3,2,29,...,49.50,160,25,14.8,17.94,1,0,1,1,1
4,CUST00119880,2024-07-12,12:39:00,12,0,0,3,3,1,49,...,30.74,134,18,38.9,2.16,0,0,0,0,0


### 3.3 Creating a new feature for identifying high risk transactions

We can create a new feature called `high_risk` for what is roughly a high risk transaction, this would be defined by
- `account_age_years` <= 1
- `time_since_last_txn_hrs` <= 1
- `is_international` == 1
- `pin_changed_recently` == 1
- `transaction_amount` >= 100

In [58]:
fraud_data['high_risk'] = np.where(
    (fraud_data['account_age_years'] <= 1) &
    (fraud_data['time_since_last_txn_hrs'] <= 1) &
    (fraud_data['is_international'] == 1) &
    (fraud_data['transaction_amount'] >= 100) &
    (fraud_data['pin_changed_recently'] == 1),
    1, 0)

filtered_df = fraud_data[fraud_data['high_risk'] == 1]

print(f"Found {filtered_df.shape[0]} high risk transactions")

fraud_data['high_risk'].describe()

Found 61 high risk transactions


count    1000000.000000
mean           0.000061
std            0.007810
min            0.000000
25%            0.000000
50%            0.000000
75%            0.000000
max            1.000000
Name: high_risk, dtype: float64